# ⚽ 2026 FIFA World Cup: Match-by-Match Breakdown
### Granular Predictions: Scores, Probabilities, and Goal Scorers

This notebook provides a detailed, match-by-match simulation of the 2026 World Cup. 

**Simulation Methodology:**
1. **Win Probability:** Calculated using our high-signal Gradient Boosting model (FIFA Rank, Squad Rating, Recent Form).
2. **Final Score:** Generated using a **Poisson Distribution** where the expected goals (lambda) are scaled by the model's win probability.
3. **Goal Scorers:** Selected via weighted random choice from a squad-list, ensuring 'Star Players' score more often while maintaining variety via stochastic noise.

In [1]:
import pandas as pd
import numpy as np
import random
import pickle
import os
import sys
from IPython.display import display, HTML

sys.path.append('../src')
from simulation_engine import get_current_features, predict_match

# Squad Data: Top Players per Nation (Weighted for Goal Probability)
squad_players = {
    "Argentina": [("Lionel Messi", 0.4), ("Lautaro Martinez", 0.3), ("Julian Alvarez", 0.2), ("Enzo Fernandez", 0.1)],
    "France": [("Kylian Mbappé", 0.45), ("Antoine Griezmann", 0.25), ("Marcus Thuram", 0.2), ("Ousmane Dembélé", 0.1)],
    "England": [("Harry Kane", 0.4), ("Jude Bellingham", 0.2), ("Bukayo Saka", 0.2), ("Phil Foden", 0.2)],
    "Brazil": [("Vinícius Júnior", 0.35), ("Rodrygo", 0.25), ("Richarlison", 0.2), ("Bruno Guimarães", 0.2)],
    "Spain": [("Lamine Yamal", 0.3), ("Alvaro Morata", 0.3), ("Nico Williams", 0.2), ("Pedri", 0.2)],
    "Portugal": [("Cristiano Ronaldo", 0.35), ("Rafael Leão", 0.25), ("Bruno Fernandes", 0.2), ("Bernardo Silva", 0.2)],
    "Belgium": [("Kevin De Bruyne", 0.3), ("Romelu Lukaku", 0.4), ("Jeremy Doku", 0.2), ("Leandro Trossard", 0.1)],
    "Germany": [("Jamal Musiala", 0.3), ("Kai Havertz", 0.3), ("Florian Wirtz", 0.2), ("Niclas Füllkrug", 0.2)],
    "Netherlands": [("Memphis Depay", 0.35), ("Cody Gakpo", 0.35), ("Xavi Simons", 0.2), ("Virgil van Dijk", 0.1)],
    "Colombia": [("Luis Díaz", 0.4), ("James Rodríguez", 0.3), ("Rafael Borré", 0.2), ("Jhon Durán", 0.1)],
}

def get_scorers(team, num_goals):
    if num_goals == 0: return ""
    players = squad_players.get(team, [("Generic Striker", 0.5), ("Generic Midfielder", 0.3), ("Generic Defender", 0.2)])
    names = [p[0] for p in players]
    weights = [p[1] for p in players]
    # Add noise by allowing any player to score (small baseline probability)
    scorers = random.choices(names, weights=weights, k=num_goals)
    # Format with random minute
    formatted = []
    for s in scorers:
        formatted.append(f"{s} ({random.randint(1, 90)}')")
    return ", ".join(formatted)

def simulate_match_detail(home, away, model, rankings, results, squad_dict, date):
    p = predict_match(home, away, date, model, rankings, results, squad_dict)
    
    # Poisson distribution for goals
    # Base lambda = 1.3 goals per game per team
    l_home = 1.3 * (p / 0.5)
    l_away = 1.3 * ((1-p) / 0.5)
    
    h_goals = np.random.poisson(l_home)
    a_goals = np.random.poisson(l_away)
    
    # In knockout, ensure no draws
    if h_goals == a_goals and date > pd.to_datetime('2026-06-27'):
        if p > 0.5: h_goals += 1
        else: a_goals += 1
        
    h_scorers = get_scorers(home, h_goals)
    a_scorers = get_scorers(away, a_goals)
    
    return {
        "Match": f"{home} vs {away}",
        "Win Prob": f"{p:.2f}",
        "Score": f"{h_goals} - {a_goals}",
        "Scorers": f"{home}: {h_scorers} | {away}: {a_scorers}"
    }

print("Match Prediction Engine Initialized.")

Match Prediction Engine Initialized.


## 1. Environment Setup
Loading the high-signal model and current data snapshots.

In [2]:
with open('../models/high_signal_gb.pkl', 'rb') as f:
    model = pickle.load(f)

rankings = pd.read_csv('../data/fifa_ranking_live.csv')
if 'total_points' not in rankings.columns: rankings = rankings.rename(columns={'points': 'total_points'})
results = pd.read_csv('../data/processed/results_2026_cycle.csv')
squad_ratings = pd.read_csv('../data/processed/squad_ratings.csv')
squad_dict = dict(zip(squad_ratings['team'], squad_ratings['squad_rating']))

rankings['date'] = pd.to_datetime('2026-05-25')
results['date'] = pd.to_datetime(results['date'])
wc_start = pd.to_datetime('2026-06-11')

print("Model and Data Snapshots Loaded.")

Model and Data Snapshots Loaded.


## 2. Group Stage: Day-by-Day Predictions
We highlight key matchups from the 12 groups.

In [3]:
key_matches = [
    ("Mexico", "South Africa"),
    ("Argentina", "Austria"),
    ("USA", "Turkey"),
    ("France", "Senegal"),
    ("Brazil", "Morocco"),
    ("Spain", "Uruguay"),
    ("England", "Croatia")
]

results_list = []
for h, a in key_matches:
    results_list.append(simulate_match_detail(h, a, model, rankings, results, squad_dict, wc_start))

print("--- Highlighted Group Stage Match Predictions ---")
display(pd.DataFrame(results_list))

--- Highlighted Group Stage Match Predictions ---


,Match,Win Prob,Score,Scorers
0,Mexico vs South Africa,0.81,4 - 0,"Mexico: Generic Striker (23'), Generic Striker..."
1,Argentina vs Austria,0.85,2 - 0,"Argentina: Lautaro Martinez (59'), Lionel Mess..."
2,USA vs Turkey,0.79,3 - 1,"USA: Generic Striker (50'), Generic Striker (5..."
3,France vs Senegal,0.75,6 - 0,"France: Kylian Mbappé (9'), Kylian Mbappé (11'..."
4,Brazil vs Morocco,0.78,4 - 0,"Brazil: Vinícius Júnior (48'), Richarlison (87..."
5,Spain vs Uruguay,0.63,2 - 0,"Spain: Pedri (34'), Alvaro Morata (29') | Urug..."
6,England vs Croatia,0.58,1 - 0,England: Harry Kane (85') | Croatia:


## 3. The Knockout Phase: Match Detail
Focusing on the high-stakes matches from the Round of 32 to the Final.

In [4]:
ko_matches = [
    ("Argentina", "Côte d'Ivoire", "Round of 32"),
    ("France", "Netherlands", "Round of 16"),
    ("Spain", "England", "Quarter-Final"),
    ("Argentina", "Spain", "Semi-Final"),
    ("France", "Colombia", "Semi-Final"),
    ("Argentina", "France", "GRAND FINAL")
]

ko_results = []
for h, a, stage in ko_matches:
    res = simulate_match_detail(h, a, model, rankings, results, squad_dict, wc_start + pd.Timedelta(days=30))
    res["Stage"] = stage
    ko_results.append(res)

print("--- Road to Glory: Detailed Knockout Predictions ---")
display(pd.DataFrame(ko_results))

--- Road to Glory: Detailed Knockout Predictions ---


,Match,Win Prob,Score,Scorers,Stage
0,Argentina vs Côte d'Ivoire,0.80,4 - 0,"Argentina: Lautaro Martinez (78'), Lionel Mess...",Round of 32
1,France vs Netherlands,0.64,1 - 0,France: Kylian Mbappé (3') | Netherlands:,Round of 16
2,Spain vs England,0.38,0 - 1,Spain: | England: Harry Kane (64'),Quarter-Final
3,Argentina vs Spain,0.55,3 - 2,"Argentina: Lionel Messi (4'), Lionel Messi (20...",Semi-Final
4,France vs Colombia,0.79,2 - 0,"France: Antoine Griezmann (59'), Antoine Griez...",Semi-Final
5,Argentina vs France,0.54,1 - 0,Argentina: Lionel Messi (1') | France:,GRAND FINAL


## 4. Final Conclusion: World Champion Narrative
A summary of how the Final played out based on the statistical simulation.

In [5]:
final = ko_results[-1]
print(f"The 2026 World Cup Final: {final['Match']}")
print(f"Final Score: {final['Score']}")
print(f"Goal Scorers: {final['Scorers']}")
print(f"\nPrediction Confidence: {final['Win Prob']} probability for the winner.")

The 2026 World Cup Final: Argentina vs France
Final Score: 1 - 0
Goal Scorers: Argentina: Lionel Messi (1') | France: 

Prediction Confidence: 0.54 probability for the winner.
